# AI应用基础 · 工单M3.2《产线实时缺陷检测系统》
**第9周 | 2课时(80分钟) | 理实一体**  | 学生实训版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 尝试batch推理或输入裁剪优化，量化提速百分比并写结论；示范讲解 |
| **B 组长** | 标准达成 | FPS≥20且检出稳定；权衡分析有理 |
| **C 操作员** | 基础完成 | CPU推理+可视化完成；阈值实验完整 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）


## 一、课前资讯（Predict·预测）
> **预习快问**：置信度阈值调高，漏检与误检各怎么变？你的机器CPU单帧约多少ms？

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m3yolo"], check=True)
print("数据已生成: data/")

## 三、任务要求（Make）
**实施步骤**
1. 加载离线权重推理≥20张并可视化
2. conf∈{0.25,0.4,0.6,0.8}阈值实验
3. imgsz三档FPS测试(warmup后计时)
4. 填写精度-速度权衡矩阵

**验收标准**：及格(D)=`CPU推理+可视化完成；阈值实验完整` ；优秀(B+)=`FPS≥20且检出稳定；权衡分析有理`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
# -*- coding: utf-8 -*-
"""M3.2 产线实时缺陷检测（预训练YOLOv8n, CPU推理主线）—— 学生代码框架
自动适配两种环境: 装了ultralytics用YOLOv8n; 否则用onnxruntime+教师导出的ONNX。
"""
import glob
import time

import cv2
import numpy as np

IMGS = sorted(glob.glob("data/yolo_dataset/images/*.npy"))
CLASSES = ["dent", "stain", "edge_crack"]
W_PATH = "data/weights/yolov8n.pt"
ONNX = "data/weights/yolov8n.onnx"

### Starter 2：`def run_ultralytics(img_bgr, conf=0.4, imgsz=3`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def run_ultralytics(img_bgr, conf=0.4, imgsz=320):
    """方案A: ultralytics"""
    from ultralytics import YOLO
    model = YOLO(W_PATH)
    r = model.predict(img_bgr, conf=conf, imgsz=imgsz, device="cpu",
                      classes=None, verbose=False)
    return r[0].plot(), len(r[0].boxes)

### Starter 3：`def run_onnx(img_bgr, conf=0.4, imgsz=320):`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def run_onnx(img_bgr, conf=0.4, imgsz=320):
    """方案B(离线备选): onnxruntime; 输入1×3×640×640, float32, /255"""
    import onnxruntime as ort
    sess = ort.InferenceSession(ONNX, providers=["CPUExecutionProvider"])
    x = cv2.resize(img_bgr, (640, 640)).astype(np.float32) / 255.0
    x = x.transpose(2, 0, 1)[None]
    out = sess.run(None, {"images": x})[0][0]        # (84,8400): 0-3=xywh, 4-83=80类分数
    # TODO-1 解析: cls = out[4:]; conf=cls.max(0), cid=cls.argmax(0); keep=conf>阈值
    # boxes = out[:4].T[keep]  (cx,cy,w,h @640, 需按原图缩放回画框)
    # === 你的代码: 统计检出数并在img_bgr画框+类别文字 ===
    return img_bgr, 0

### Starter 4：`def pick_backend():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def pick_backend():
    try:
        import ultralytics  # noqa
        return "ultralytics"
    except ImportError:
        return "onnx"

### Starter 5：`def main():`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
def main():
    backend = pick_backend()
    runner = run_ultralytics if backend == "ultralytics" else run_onnx
    print("后端:", backend)

    # ---- 1) 推理可视化(前5张) ----
    for p in IMGS[:5]:
        img = np.load(p).astype(np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        anno, n = runner(img)
        cv2.imwrite("yolo_" + p.split("/")[-1].replace(".npy", ".png"), anno)
        print(p.split("/")[-1], "检出", n)

    # ---- 2) 阈值实验: conf∈{0.25,0.4,0.6,0.8} ----
    # TODO-2: 循环调用runner, 统计20张图总检出数, 填入工单表格
    # ---- 3) FPS测试: imgsz∈{160,320,640} ----
    # TODO-3: warmup 3帧后计时20帧, 打印平均FPS(只计推理段!)
    # for imgsz in (160, 320, 640):
    #     t0 = time.time()
    #     for p in IMGS[:20]: ...
    #     fps = 20 / (time.time() - t0)
    # ---- 4) 决策矩阵: 综合检出与FPS选择产线配置 ----

### Starter 6：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [ ]:
if __name__ == "__main__":
    main()

## 四、提交要求与避坑
- 提交: `学号_姓名_M3.2.py` 或本notebook(含输出)
- ⚠ 权重从data/weights离线加载，禁止课堂在线下载
- ⚠ FPS计时只包推理段：warmup 3帧后再计时
- ⚠ ONNX输出是(84,8400)：0-3行xywh，4-83行类别分